# 112 — Pairwise Blend Optimizer

Exhaustive grid search over all legitimate OOF pairs (excluding leaky features).
Finds the best 2-model, 3-model, and rank-weighted ensemble.

Goal: find combinations that beat nb109 OOF RAE = 0.2748.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8")
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from itertools import combinations
from pathlib import Path
from scipy.optimize import minimize
from pxr.data import load_train, load_test
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko
from pxr.paths import DATA_PROCESSED, SUBMISSIONS
SEED = 42; N_FOLDS = 5
print("imports OK")

imports OK


In [2]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()

# Load all OOF files and filter to legitimate ones (RAE < 0.60, n_valid >= 3800)
# Exclude known leaky files (emax/pec50_se features)
LEAKY = {
    "oof_aux_features.npy",
    "oof_creative_mega_ensemble.npy",
    "oof_grand_v6.npy",
    "oof_grand_v7.npy",
    "oof_grand_v5.npy",
}

all_oof = {}
for f in sorted(DATA_PROCESSED.glob("oof_*.npy")):
    if f.name in LEAKY:
        continue
    try:
        arr = np.load(f)
        if len(arr) != len(y_tr): continue
        mask = np.isfinite(arr)
        if mask.sum() < 3800: continue
        r = rae(y_tr[mask], arr[mask])
        if r > 0.65: continue  # skip very weak models
        all_oof[f.stem.replace("oof_", "")] = arr
    except: pass

print(f"Loaded {len(all_oof)} legitimate OOF arrays")

names = sorted(all_oof.keys(), key=lambda n: rae(y_tr, all_oof[n]))
print("\nTop-20 individual OOF RAEs:")
for n in names[:20]:
    r = rae(y_tr, all_oof[n])
    print(f"  {r:.4f}  {n}")

Loaded 109 legitimate OOF arrays

Top-20 individual OOF RAEs:
  0.2480  enhanced_delta_3tier
  0.2748  grand_v9
  0.2748  delta_ensemble_blend
  0.2772  delta_similarity_tiers
  0.2843  grand_v8
  0.2888  delta_5tiers
  0.3266  delta_loso
  0.3266  multi_template_delta
  0.3268  delta_uncertainty
  0.3269  reverse_delta_ml
  0.3693  nb125_2way
  0.3706  nb119_optuna_ensemble
  0.3714  nb112_grand_v3
  0.3715  nb125_enet
  0.3734  nb109_deep_meta_stack_calib
  0.3741  nb109_deep_meta_stack
  0.3741  nb117_knn_residual
  0.3741  nb124_scaffold_specific
  0.3748  nb125_lgbm_meta
  0.3754  nb111_selectivity_primary


In [3]:
# Load corresponding test predictions
all_te = {}
for name in all_oof:
    te_path = DATA_PROCESSED / f"te_oof_{name}.npy"
    if te_path.exists():
        arr = np.load(te_path)
        if len(arr) == len(te): all_te[name] = arr

print(f"Test predictions available for {len(all_te)}/{len(all_oof)} models")

Test predictions available for 37/109 models


In [4]:
# --- Exhaustive 2-model optimal blend ---
print("\n=== Exhaustive 2-model optimal blend ===", flush=True)
best2_results = []

for n1, n2 in combinations(names, 2):
    oof1, oof2 = all_oof[n1], all_oof[n2]
    # Search optimal alpha in [0, 1]
    best_r, best_alpha = float("inf"), 0.5
    for alpha in np.linspace(0, 1, 21):
        blended = alpha * oof1 + (1 - alpha) * oof2
        mask = np.isfinite(blended)
        r = rae(y_tr[mask], blended[mask])
        if r < best_r:
            best_r, best_alpha = r, alpha
    best2_results.append((best_r, best_alpha, n1, n2))

best2_results.sort()
print("Top-20 2-model blends:")
for r, alpha, n1, n2 in best2_results[:20]:
    print(f"  RAE={r:.4f}  alpha={alpha:.2f}  {n1} + {n2}")


=== Exhaustive 2-model optimal blend ===


Top-20 2-model blends:
  RAE=0.2473  alpha=0.95  enhanced_delta_3tier + delta_ensemble_blend
  RAE=0.2473  alpha=0.95  enhanced_delta_3tier + grand_v9
  RAE=0.2480  alpha=1.00  enhanced_delta_3tier + 3d_shape
  RAE=0.2480  alpha=1.00  enhanced_delta_3tier + 3d_shape_conformer
  RAE=0.2480  alpha=1.00  enhanced_delta_3tier + all_feature_fusion
  RAE=0.2480  alpha=1.00  enhanced_delta_3tier + bio_nr_fingerprint
  RAE=0.2480  alpha=1.00  enhanced_delta_3tier + catboost
  RAE=0.2480  alpha=1.00  enhanced_delta_3tier + chemberta_esm2_nr
  RAE=0.2480  alpha=1.00  enhanced_delta_3tier + chemberta_mtr
  RAE=0.2480  alpha=1.00  enhanced_delta_3tier + chemprop_aux
  RAE=0.2480  alpha=1.00  enhanced_delta_3tier + chemprop_cliff
  RAE=0.2480  alpha=1.00  enhanced_delta_3tier + cliff_adaptive_blend
  RAE=0.2480  alpha=1.00  enhanced_delta_3tier + cliff_weighted
  RAE=0.2480  alpha=1.00  enhanced_delta_3tier + consensus_delta_ml
  RAE=0.2480  alpha=1.00  enhanced_delta_3tier + crossattn_chemberta_es

In [5]:
# --- Top-3 model blends (using top-30 individual models) ---
print("\n=== Top-15 3-model blends (from top-30 individuals) ===", flush=True)
top30 = names[:30]
best3_results = []

for n1, n2, n3 in combinations(top30, 3):
    oof1, oof2, oof3 = all_oof[n1], all_oof[n2], all_oof[n3]
    # Equal weight first
    blended = (oof1 + oof2 + oof3) / 3
    mask = np.isfinite(blended)
    r_eq = rae(y_tr[mask], blended[mask])
    best3_results.append((r_eq, 1/3, 1/3, 1/3, n1, n2, n3))

best3_results.sort()
print("Top-15 3-model equal-weight blends:")
for r, w1, w2, w3, n1, n2, n3 in best3_results[:15]:
    print(f"  RAE={r:.4f}  {n1} + {n2} + {n3}")


=== Top-15 3-model blends (from top-30 individuals) ===


Top-15 3-model equal-weight blends:
  RAE=0.2590  enhanced_delta_3tier + grand_v9 + delta_similarity_tiers
  RAE=0.2591  enhanced_delta_3tier + delta_ensemble_blend + delta_similarity_tiers
  RAE=0.2597  enhanced_delta_3tier + grand_v9 + delta_5tiers
  RAE=0.2597  enhanced_delta_3tier + delta_ensemble_blend + delta_5tiers
  RAE=0.2598  enhanced_delta_3tier + grand_v9 + delta_ensemble_blend
  RAE=0.2601  enhanced_delta_3tier + delta_ensemble_blend + grand_v8
  RAE=0.2605  enhanced_delta_3tier + grand_v9 + grand_v8
  RAE=0.2615  enhanced_delta_3tier + grand_v8 + delta_5tiers
  RAE=0.2618  enhanced_delta_3tier + delta_similarity_tiers + delta_5tiers
  RAE=0.2646  enhanced_delta_3tier + delta_similarity_tiers + grand_v8
  RAE=0.2674  enhanced_delta_3tier + grand_v9 + delta_loso
  RAE=0.2674  enhanced_delta_3tier + delta_ensemble_blend + delta_loso
  RAE=0.2675  enhanced_delta_3tier + grand_v9 + multi_template_delta
  RAE=0.2675  enhanced_delta_3tier + delta_ensemble_blend + multi_template_

In [6]:
# --- Rank-weighted ensemble of top-N models ---
print("\n=== Rank-weighted ensemble of top-N models ===", flush=True)
results_topN = []
for N in [2, 3, 4, 5, 7, 10, 15, 20]:
    top_n = names[:N]
    raes_n = np.array([rae(y_tr, all_oof[n]) for n in top_n])
    # inverse RAE weights
    weights = 1.0 / np.maximum(raes_n, 1e-6)
    weights /= weights.sum()
    blended = sum(w * all_oof[n] for w, n in zip(weights, top_n))
    mask = np.isfinite(blended)
    r = rae(y_tr[mask], blended[mask])
    results_topN.append((N, r, weights))
    print(f"  top-{N:2d}  RAE={r:.4f}  best_w={weights[0]:.3f}")

best_N = min(results_topN, key=lambda x: x[1])
print(f"\nBest rank-weighted: top-{best_N[0]} RAE={best_N[1]:.4f}")


=== Rank-weighted ensemble of top-N models ===


  top- 2  RAE=0.2539  best_w=0.526
  top- 3  RAE=0.2590  best_w=0.357
  top- 4  RAE=0.2604  best_w=0.270
  top- 5  RAE=0.2621  best_w=0.219
  top- 7  RAE=0.2640  best_w=0.162
  top-10  RAE=0.2763  best_w=0.118
  top-15  RAE=0.2914  best_w=0.085
  top-20  RAE=0.3047  best_w=0.066

Best rank-weighted: top-2 RAE=0.2539


In [7]:
# --- Optimal blend via scipy optimize for top-5 ---
from scipy.optimize import minimize as sp_minimize

print("\n=== Scipy-optimized weights for top-5 models ===", flush=True)
top5 = names[:5]
OOF5 = np.column_stack([all_oof[n] for n in top5])

def neg_rae(w):
    w = np.abs(w); w /= w.sum()
    blended = OOF5 @ w
    mask = np.isfinite(blended)
    return rae(y_tr[mask], blended[mask])

best_r_opt, best_w_opt = float("inf"), None
for _ in range(20):  # multiple restarts
    w0 = np.random.dirichlet(np.ones(5))
    res = sp_minimize(neg_rae, w0, method="Nelder-Mead",
                      options={"maxiter": 5000, "xatol": 1e-6})
    if res.fun < best_r_opt:
        best_r_opt, best_w_opt = res.fun, res.x

best_w_opt = np.abs(best_w_opt); best_w_opt /= best_w_opt.sum()
print("Optimized weights (top-5):")
for n, w in zip(top5, best_w_opt):
    print(f"  {w:.4f}  {n}")
print(f"Optimized OOF RAE: {best_r_opt:.4f}")


=== Scipy-optimized weights for top-5 models ===


Optimized weights (top-5):
  0.9363  enhanced_delta_3tier
  0.0178  grand_v9
  0.0459  delta_ensemble_blend
  0.0000  delta_similarity_tiers
  0.0000  grand_v8
Optimized OOF RAE: 0.2473


In [8]:
# --- Summary and best submission ---
print("\n=== Summary ===")
best_2m = best2_results[0]
best_3m = best3_results[0]
best_N_val = best_N[1]
best_opt = best_r_opt

all_approaches = [
    ("best_2model",  best_2m[0]),
    ("best_3model",  best_3m[0]),
    (f"rank_top{best_N[0]}", best_N_val),
    ("opt_top5",     best_opt),
    # Reference: individual models
    (names[0],       rae(y_tr, all_oof[names[0]])),
]
all_approaches.sort(key=lambda x: x[1])
for name, r in all_approaches:
    print(f"  {r:.4f}  {name}")

# Save best approach
best_name, best_rae_v = all_approaches[0]
print(f"\nBest: {best_name}  RAE={best_rae_v:.4f}")

# Build best blend predictions
if "opt_top5" in best_name:
    OOF_final = OOF5 @ best_w_opt
    if all(n in all_te for n in top5):
        TE_final = np.column_stack([all_te[n] for n in top5]) @ best_w_opt
    else:
        TE_final = None
elif "2model" in best_name:
    r, alpha, n1, n2 = best_2m
    OOF_final = alpha * all_oof[n1] + (1-alpha) * all_oof[n2]
    TE_final = (alpha * all_te.get(n1, np.zeros(len(te))) +
                (1-alpha) * all_te.get(n2, np.zeros(len(te)))) if n1 in all_te and n2 in all_te else None
else:
    OOF_final = all_oof[names[0]]  # best single
    TE_final = all_te.get(names[0])

print(f"OOF RAE = {rae(y_tr[np.isfinite(OOF_final)], OOF_final[np.isfinite(OOF_final)]):.4f}")

if TE_final is not None:
    TE_final = np.clip(TE_final, y_tr.min()-0.5, y_tr.max()+0.5)
    np.save(DATA_PROCESSED/"oof_blend_optimizer.npy", OOF_final)
    np.save(DATA_PROCESSED/"te_oof_blend_optimizer.npy", TE_final)
    sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": TE_final})
    assert len(sub)==513 and sub["pEC50"].notna().all()
    p = SUBMISSIONS/"112_pairwise_blend_optimizer.csv"; sub.to_csv(p, index=False)
    print(f"Saved {p}")
    print(f"Test: min={TE_final.min():.2f} med={np.median(TE_final):.2f} max={TE_final.max():.2f}")

print(f"\n*** nb112 OOF RAE = {rae(y_tr[np.isfinite(OOF_final)], OOF_final[np.isfinite(OOF_final)]):.4f} ***")


=== Summary ===
  0.2473  opt_top5
  0.2473  best_2model
  0.2480  enhanced_delta_3tier
  0.2539  rank_top2
  0.2590  best_3model

Best: opt_top5  RAE=0.2473
OOF RAE = 0.2473
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\112_pairwise_blend_optimizer.csv
Test: min=3.29 med=4.90 max=6.54

*** nb112 OOF RAE = 0.2473 ***
